# 04 — Réévaluation complète sur l'export MongoDB

Ce notebook refait toute la chaîne sur l'export Compass `yele_speedtest.speedtest_results.csv` :

1. chargement et nettoyage (`prepare_data.py`)
2. description du jeu de données (tableau 5.3, figure 5.10)
3. résultats par opérateur (tableau 5.4, figure 5.11)
4. tests d'usage et avis des utilisateurs (tableau 5.5, Spearman)
5. comparaison des détecteurs d'anomalies (tableau 5.6)
6. alertes levées par le service (tableau 5.7)
7. prédiction du débit (tableau 5.8)
8. récapitulatif des valeurs à reporter dans le rapport

Choisir le noyau **.venv**, puis **Exécuter tout**. TabPFN (section 7) prend quelques minutes.

## 0. Paramètres

In [ ]:
import os, warnings
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import prepare_data as P

SEUIL_OPERATEUR = 30        # mesures minimum pour figurer dans le tableau 5.4
RESEAUX_MOBILES = ["Mobile", "4G", "3G", "2G", "5G"]
# Moov Africa est la marque commerciale d'ONATEL : on les regroupe.
FUSION_OPERATEURS = {"ONATEL / Telmob": "Moov Africa (ONATEL)",
                     "Moov Africa": "Moov Africa (ONATEL)"}
# Appareils qui ne sont pas des téléphones (exclus du comptage des modèles)
PAS_TELEPHONE = ["PC Windows 10", "PC Windows 11", "Tablette Android", "MacBook Pro",
                 "CHROME", "EDGE", "Android SDK built for x86_64", "Inconnu"]

fr = lambda v, d=2: f"{v:,.{d}f}".replace(",", " ").replace(".", ",")
RESUME = {}   # valeurs à reporter, remplies au fil du notebook

## 1. Chargement et nettoyage

In [ ]:
brut = P.load_raw()
df = P.clean(brut.copy())
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, format="mixed")
df.to_csv(P.OUTPUT, index=False)   # lu par evaluate.py et les autres scripts

RESUME["Mesures brutes"] = len(brut)
RESUME["Mesures écartées"] = len(brut) - len(df)
RESUME["Jeu final"] = len(df)
print(f"\n{len(brut) - len(df)} mesures écartées, jeu final de {len(df)} mesures")

## 2. Description du jeu de données (tableau 5.3)

In [ ]:
telephones = df.loc[~df.device_model.isin(PAS_TELEPHONE), "device_model"].nunique()
ville = df.loc[df.city != "Inconnu", "city"].value_counts()
part_mobile = df.network_type.isin(RESEAUX_MOBILES).mean() * 100
part_wifi = df.is_wifi.mean() * 100

RESUME["Modèles de téléphones"] = telephones
RESUME["Ville principale"] = f"{ville.index[0]} ({ville.iloc[0] / len(df) * 100:.0f} %)"
RESUME["Part mobile (%)"] = round(part_mobile)
RESUME["Part WiFi (%)"] = round(part_wifi)
print(f"Modèles de téléphones : {telephones}")
print(f"Ville principale      : {RESUME['Ville principale']}")
print(f"Réseau mobile {part_mobile:.0f} %, WiFi {part_wifi:.0f} %, autre {100 - part_mobile - part_wifi:.0f} %")

In [ ]:
noms = {"dl": "Débit descendant (Mbit/s)", "ul": "Débit montant (Mbit/s)",
        "ping": "Latence (ms)", "jitter": "Gigue (ms)"}
t53 = pd.DataFrame({noms[c]: [df[c].median(), df[c].mean(), df[c].min(), df[c].max()]
                    for c in noms}, index=["Médiane", "Moyenne", "Minimum", "Maximum"]).T
t53.map(fr)

### Figure 5.10 — répartition par classe de débit (grille du tableau 4.10)

In [ ]:
bornes = [0, 3, 10, 20, 30, 50, 100, np.inf]
etiquettes = ["< 3", "3 à 10", "10 à 20", "20 à 30", "30 à 50", "50 à 100", "100 et plus"]
couleurs = ["#DE7672", "#FBC85F", "#A0DBA0", "#A0DBA0", "#55BE55", "#55BE55", "#55BE55"]
classes = pd.cut(df.dl, bornes, labels=etiquettes, right=False).value_counts().reindex(etiquettes)
pct = classes / len(df) * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(etiquettes, classes, color=couleurs, width=0.75)
for i, (n, p) in enumerate(zip(classes, pct)):
    ax.text(i, n + 2, f"{n} ({p:.0f} %)", ha="center")
ax.set_yticks([]); [ax.spines[s].set_visible(False) for s in ["top", "right", "left"]]
ax.set_title(f"{pct.iloc[4:].sum():.0f} % des mesures dépassent 30 Mbit/s, "
             f"{pct.iloc[0]:.0f} % restent sous 3 Mbit/s", loc="left")
plt.show()
RESUME["Part > 30 Mbit/s (%)"] = round(pct.iloc[4:].sum())
RESUME["Part < 3 Mbit/s (%)"] = round(pct.iloc[0])

## 3. Résultats par opérateur (tableau 5.4, figure 5.11)

In [ ]:
mob = df[df.network_type.isin(RESEAUX_MOBILES)].copy()
mob["op"] = mob.operator.replace(FUSION_OPERATEURS)
q1 = lambda s: s.quantile(0.25)
q3 = lambda s: s.quantile(0.75)
t54 = (mob.groupby("op")
          .agg(Mesures=("dl", "size"), DL=("dl", "median"), UL=("ul", "median"),
               Latence=("ping", "median"), Gigue=("jitter", "median"),
               Q1=("dl", q1), Q3=("dl", q3))
          .query("Mesures >= @SEUIL_OPERATEUR")
          .sort_values("DL", ascending=False))
RESUME["Débit mobile médian"] = fr(mob[mob.op.isin(t54.index)].dl.median(), 1)
t54.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ys = list(range(len(t54)))[::-1]
for y, (op, r) in zip(ys, t54.iterrows()):
    c = "#2E7AD6" if y == max(ys) else "#C4C4B8"
    ax.plot([r.Q1, r.Q3], [y, y], color=c, lw=3.5)
    ax.plot(r.DL, y, "o", color=c, ms=12)
    ax.text(r.DL, y + 0.2, fr(r.DL, 1), ha="center")
ax.set_yticks(ys); ax.set_yticklabels([f"{op} ({int(r.Mesures)})" for op, r in t54.iterrows()])
ax.grid(axis="x", color="#e3e3e3"); [ax.spines[s].set_visible(False) for s in ax.spines]
ax.set_title("Débit descendant mobile (Mbit/s) : médiane et 1er-3e quartile", loc="left")
plt.show()

In [ ]:
# Variation du débit selon l'heure. Le Burkina Faso est en UTC+0.
tranches = pd.cut(df.timestamp.dt.hour, [-1, 5, 11, 17, 23],
                  labels=["Nuit 0 h-6 h", "Matin 6 h-12 h", "Après-midi 12 h-18 h", "Soir 18 h-24 h"])
df.groupby(tranches, observed=False).dl.agg(Mesures="size", Médiane="median").round(1)

## 4. Tests d'usage et avis des utilisateurs (tableau 5.5)

In [ ]:
st = df[df.streaming_score > 0]
br = df[df.browsing_score > 0]
av = df[df.qoe_rating > 0]

hd = st.streaming_max_resolution.isin(["1080p", "1440p", "2160p"]).mean() * 100
t55 = pd.DataFrame([
    ["Streaming vidéo", len(st), f"Score de {fr(st.streaming_score.median(), 1)} sur 100",
     f"Aucune interruption dans {(st.streaming_rebuffer_count == 0).mean() * 100:.0f} % des tests. "
     f"Résolution de 1080p ou plus dans {hd:.0f} % des tests"],
    ["Navigation web", len(br), f"Score de {fr(br.browsing_score.median(), 1)} sur 100",
     f"Chargement médian de {fr(br.browsing_avg_load_ms.median() / 1000, 1)} s. "
     f"Toutes les pages chargées dans {(br.browsing_success_rate >= 1).mean() * 100:.0f} % des tests"],
    ["Avis des utilisateurs", len(av), f"Note de {fr(av.qoe_rating.median(), 0)} sur 5",
     f"{av.qoe_sat.isin(['satisfied', 'very_satisfied']).mean() * 100:.0f} % d’utilisateurs "
     f"satisfaits ou très satisfaits"],
], columns=["Test", "Tests exploitables", "Résultat médian", "Observations"])
pd.set_option("display.max_colwidth", None)
t55

In [ ]:
rho_dl = spearmanr(av.qoe_rating, av.dl).statistic
rho_ping = spearmanr(av.qoe_rating, av.ping).statistic
RESUME["Spearman note / débit"] = fr(rho_dl)
RESUME["Spearman note / latence"] = fr(rho_ping)
print(f"Spearman note / débit   : {rho_dl:+.2f}")
print(f"Spearman note / latence : {rho_ping:+.2f}")

## 5. Comparaison des détecteurs d'anomalies (tableau 5.6)

Protocole de `pyod_detectors.py` : apprentissage sur les 70 % de mesures les plus anciennes,
15 % du jeu de test dégradé volontairement, 20 répétitions. Tous les détecteurs reçoivent
la même préparation que le service : KPI passés au log puis mis à l'échelle.

In [ ]:
import pyod_detectors as PD

t56 = PD.compare(df)
RESUME["Meilleur détecteur"] = t56.iloc[0]["Détecteur"]
print(f"Référence du hasard pour la PR-AUC : {PD.INJECTION_RATE:.3f}")
t56

### Stabilité du classement sur cinq graines d'injection

In [ ]:
res = []
for g in [1, 7, 42, 123, 2026]:
    PD.SEED = g
    res.append(PD.compare(df, n_repeats=10))
PD.SEED = 42
(pd.concat(res).groupby("Détecteur")[["ROC-AUC", "PR-AUC", "F1 (max)"]]
   .agg(["mean", "std"]).round(3).sort_values(("ROC-AUC", "mean"), ascending=False))

## 6. Alertes levées par le service (tableau 5.7)

In [ ]:
from model import YeleAI

ai = YeleAI()
ai.train(df.sort_values("timestamp").reset_index(drop=True))
alertes = pd.DataFrame(ai.detect(df, limit=len(df)))
alertes["reasons"] = alertes["reasons"].map(" | ".join)
RESUME["Alertes levées"] = len(alertes)
print(f"Détecteur : {ai.detector_name} · prédiction : {ai.reg_name}")
print(f"{len(alertes)} alertes levées\n")
alertes[["operator", "networkType", "download", "ping", "severity", "reasons"]].head(15)

## 7. Prédiction du débit (tableau 5.8)

Découpage chronologique 80/20, puis cinq découpages successifs pour vérifier le classement.
Le skill score compare l'erreur du modèle à celle de la médiane (0 = pas mieux que la médiane).

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from comparaison_approches import make_tabpfn, FOLDS

d = df.sort_values("timestamp").reset_index(drop=True)
enc = YeleAI()
enc.operators = sorted(d.operator.unique()); enc.networks = sorted(d.network_type.unique())
x, y = enc._features(d), d.dl.to_numpy()
modeles = {"Gradient Boosting": lambda: HistGradientBoostingRegressor(random_state=42),
           "TabPFN": make_tabpfn}

coupe = int(len(d) * 0.8)
mae_ref = np.mean(np.abs(y[coupe:] - np.median(y[:coupe])))
lignes = [["Référence naïve (médiane)", mae_ref, 0.0, 0.0]]
for nom, fabrique in modeles.items():
    t0 = time.time()
    pred = fabrique().fit(x[:coupe], y[:coupe]).predict(x[coupe:])
    mae = np.mean(np.abs(y[coupe:] - pred))
    lignes.append([nom, mae, 1 - mae / mae_ref, time.time() - t0])
t58 = pd.DataFrame(lignes, columns=["Modèle", "MAE (Mbit/s)", "Skill score", "Durée (s)"])
print(f"{coupe} mesures d'apprentissage, {len(d) - coupe} de test")
t58.round(3)

In [ ]:
plis = []
for a, b in FOLDS:
    i, j = int(len(d) * a), int(len(d) * b)
    ref = np.mean(np.abs(y[i:j] - np.median(y[:i])))
    ligne = {"Pli": f"{int(a * 100)}-{int(b * 100)} %"}
    for nom, fabrique in modeles.items():
        mae = np.mean(np.abs(y[i:j] - fabrique().fit(x[:i], y[:i]).predict(x[i:j])))
        ligne[nom] = 1 - mae / ref
    plis.append(ligne)
plis = pd.DataFrame(plis).set_index("Pli")
victoires = int((plis["Gradient Boosting"] > plis["TabPFN"]).sum())
RESUME["Gradient Boosting meilleur sur"] = f"{victoires} plis sur {len(plis)}"
plis.loc["Moyenne"] = plis.mean()
plis.round(3)

## 8. Récapitulatif des valeurs à reporter dans le rapport

In [ ]:
pd.Series(RESUME, name="Valeur").to_frame()